# 08 — Category Scorecard

**Purpose:** Birds-eye view of every category in the portfolio. Growth trends, churn exposure, segment mix, top players.  
**Key question:** Which categories are healthy, which are declining, and where should we focus?  

**Tables used:** `marts.mart_category_scorecard`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Full scorecard
Every category with spend, growth, churn risk, and segment mix.

In [ ]:
scorecard = q(f"""
    SELECT CATEGORY_TWO, num_destinations, total_customers, total_spend,
           avg_txn_value, avg_spend_per_customer,
           spend_recent_3m, spend_prior_3m, growth_pct,
           top_destination_name, top_dest_market_share,
           avg_churn_pct, high_risk_customers, high_risk_pct,
           pct_champions, pct_dormant, health_status
    FROM `{PROJECT}.marts.mart_category_scorecard`
    ORDER BY total_spend DESC
""")

if not scorecard.empty:
    print(f'{len(scorecard)} categories scored')
    display(scorecard)
else:
    print('No data. Run mart_category_scorecard SQL first.')

## 2. Health overview
How many categories are growing, stable, slowing, or declining?

In [ ]:
if not scorecard.empty:
    health_colors = {'Growing': '#4caf50', 'Stable': '#2196f3', 'Slowing': '#ff9800', 'Declining': '#f44336'}
    health = scorecard.groupby('health_status').agg(
        categories=('CATEGORY_TWO', 'count'),
        total_spend=('total_spend', 'sum')
    ).reset_index()

    fig = px.pie(health, values='categories', names='health_status',
                 color='health_status', color_discrete_map=health_colors,
                 title='Category health distribution')
    fig.show()

    print('\nBreakdown:')
    for _, row in health.iterrows():
        print(f'  {row["health_status"]}: {row["categories"]} categories, R{row["total_spend"]:,.0f} total spend')

## 3. Growth vs churn risk
Categories in the top-right (high growth, low churn) are the healthiest. Bottom-left = trouble.

In [ ]:
if not scorecard.empty:
    sc = scorecard[scorecard['growth_pct'].notna()].copy()
    if not sc.empty:
        fig = px.scatter(sc, x='growth_pct', y='avg_churn_pct',
                         size='total_spend', color='health_status',
                         color_discrete_map={'Growing': '#4caf50', 'Stable': '#2196f3', 'Slowing': '#ff9800', 'Declining': '#f44336'},
                         hover_name='CATEGORY_TWO',
                         title='Category growth vs churn risk (size = total spend)',
                         labels={'growth_pct': 'Growth % (3m vs prior 3m)', 'avg_churn_pct': 'Avg churn risk %'})
        fig.add_hline(y=sc['avg_churn_pct'].median(), line_dash='dash', line_color='gray', opacity=0.5)
        fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)
        fig.update_layout(height=600)
        fig.show()

## 4. Top growing categories

In [ ]:
if not scorecard.empty:
    growing = scorecard[scorecard['growth_pct'].notna()].nlargest(10, 'growth_pct')
    if not growing.empty:
        fig = px.bar(growing, x='growth_pct', y='CATEGORY_TWO', orientation='h',
                     color='total_spend', color_continuous_scale='Greens',
                     title='Top 10 fastest growing categories',
                     labels={'growth_pct': 'Growth %', 'CATEGORY_TWO': ''})
        fig.update_layout(yaxis=dict(autorange='reversed'))
        fig.show()

## 5. Most at-risk categories
Highest churn exposure — these need attention.

In [ ]:
if not scorecard.empty:
    risky = scorecard.nlargest(10, 'high_risk_pct')
    if not risky.empty:
        fig = px.bar(risky, x='high_risk_pct', y='CATEGORY_TWO', orientation='h',
                     color='total_spend', color_continuous_scale='Reds',
                     title='Top 10 categories by churn risk exposure',
                     labels={'high_risk_pct': '% customers at high/critical risk', 'CATEGORY_TWO': ''})
        fig.update_layout(yaxis=dict(autorange='reversed'))
        fig.show()

## 6. Segment composition
What % of each category is Champions vs Dormant?

In [ ]:
if not scorecard.empty:
    top20 = scorecard.nlargest(20, 'total_spend')[['CATEGORY_TWO', 'pct_champions', 'pct_dormant']].copy()

    fig = go.Figure()
    fig.add_trace(go.Bar(name='Champions %', y=top20['CATEGORY_TWO'], x=top20['pct_champions'],
                         orientation='h', marker_color='#2E75B6'))
    fig.add_trace(go.Bar(name='Dormant %', y=top20['CATEGORY_TWO'], x=top20['pct_dormant'],
                         orientation='h', marker_color='#f44336'))
    fig.update_layout(barmode='group', title='Champions vs Dormant mix — top 20 categories by spend',
                      xaxis_title='% of customers', height=600,
                      yaxis=dict(autorange='reversed'))
    fig.show()